# STEP 5. 결과 시각화
- 그래프 1: 관측소별 누적 DD + 위험도 구간
- 그래프 2: Sigmoid 개체군 곡선 (1화기·2화기)
- 그래프 3: 연도별 1화기 예보 발생일 비교

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

SAVE_DIR   = '/content/drive/MyDrive/JADX_병해충/data'
RESULT_FILE = f'{SAVE_DIR}/result_pest_model.csv'

df = pd.read_csv(RESULT_FILE)
df['date'] = pd.to_datetime(df['crtr_ymd'], format='%Y%m%d')

print(f'✅ 결과 데이터 로드: {len(df)}건')

In [ ]:
# ── 그래프 1: 연도별 누적 DD + 위험도 구간 (관측소별) ────────
# 연도 선택
TARGET_YEAR = 2024   # ← 원하는 연도로 변경 가능

df_year = df[df['year'] == TARGET_YEAR].copy()
stations = df_year['stn_nm'].unique()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

# 위험도 구간 색상
ZONE_COLORS = [
    (0,    188.7, '#E8F5E9', '보통'),
    (188.7,240.0, '#FFF9C4', '1화기 주의'),
    (240.0,289.4, '#FFE0B2', '1화기 경보'),
    (289.4,745.0, '#FFCDD2', '1화기 심각'),
    (745.0,883.7, '#FFF9C4', '2화기 주의'),
    (883.7,984.4, '#FFE0B2', '2화기 경보'),
    (984.4,1500,  '#FFCDD2', '2화기 심각'),
]

for i, stn in enumerate(sorted(stations)):
    ax = axes[i]
    grp = df_year[df_year['stn_nm'] == stn].sort_values('jld')

    # 위험도 구간 배경색
    for low, high, color, label in ZONE_COLORS:
        ax.axhspan(low, high, alpha=0.3, color=color)

    # 누적 DD 선
    ax.plot(grp['jld'], grp['cumdd'], color='#1565C0', linewidth=2, label='누적 DD')

    # 위험도 임계값 수평선
    ax.axhline(y=188.7, color='orange', linestyle='--', linewidth=1, alpha=0.8, label='1화기 주의')
    ax.axhline(y=745.0, color='red',    linestyle='--', linewidth=1, alpha=0.8, label='2화기 주의')

    # 논문 권장 범위 세로선
    ax.axvline(x=120, color='green', linestyle=':', linewidth=1.5, alpha=0.7, label='권장범위 시작(J120)')
    ax.axvline(x=135, color='green', linestyle=':', linewidth=1.5, alpha=0.7, label='권장범위 끝(J135)')

    # 실제 주의 발령일 표시
    hit = grp[grp['cumdd'] >= 188.7]
    if len(hit) > 0:
        r = hit.iloc[0]
        ax.axvline(x=r['jld'], color='red', linewidth=2, alpha=0.9)
        ax.annotate(f"1화기 주의\n{r['date'].strftime('%m/%d')}(J{int(r['jld'])})",
                    xy=(r['jld'], 188.7), xytext=(r['jld']+5, 250),
                    fontsize=9, color='red',
                    arrowprops=dict(arrowstyle='->', color='red', lw=1))

    ax.set_title(f'{stn} ({TARGET_YEAR}년)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Julian Date')
    ax.set_ylabel('누적 DD')
    ax.set_xlim(1, 365)
    ax.set_ylim(0, 1200)
    ax.set_xticks([1,60,120,180,240,300,365])
    ax.set_xticklabels(['1/1','3/1','5/1','7/1','9/1','11/1','12/31'], fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'네눈쑥가지나방 누적 적산온도 및 위험도 구간 ({TARGET_YEAR}년)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/graph1_cumdd_{TARGET_YEAR}.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 그래프 1 저장 완료')

In [ ]:
# ── 그래프 2: Sigmoid 개체군 곡선 ──────────────────────────
# 관측소·연도 선택
TARGET_STN  = '제주'   # ← 원하는 관측소로 변경 가능
TARGET_YEAR = 2024

grp = df[(df['stn_nm']==TARGET_STN) & (df['year']==TARGET_YEAR)].sort_values('jld')

fig, ax = plt.subplots(figsize=(14, 6))

# Sigmoid 곡선
ax.plot(grp['jld'], grp['sigmoid_1화기'], color='#1565C0', linewidth=2.5, label='1화기 개체군')
ax.plot(grp['jld'], grp['sigmoid_2화기'], color='#C62828', linewidth=2.5, label='2화기 개체군')

# 10% / 50% / 90% 기준선
for pct, ls, label in [(0.1,'--','10% (주의)'), (0.5,'-.','50% (경보)'), (0.9,':','90% (심각)')]:
    ax.axhline(y=pct, color='gray', linestyle=ls, linewidth=1, alpha=0.7, label=label)

# 논문 성충 피크 범위
ax.axvspan(139, 145, alpha=0.15, color='green', label='논문 1화기 성충피크(J139~145)')

ax.set_xlabel('Julian Date')
ax.set_ylabel('개체군 출현율 (0~1)')
ax.set_title(f'네눈쑥가지나방 Sigmoid 개체군 곡선 ({TARGET_STN}, {TARGET_YEAR}년)', fontsize=13, fontweight='bold')
ax.set_xlim(1, 365)
ax.set_ylim(-0.05, 1.05)
ax.set_xticks([1,60,120,180,240,300,365])
ax.set_xticklabels(['1/1','3/1','5/1','7/1','9/1','11/1','12/31'])
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/graph2_sigmoid_{TARGET_STN}_{TARGET_YEAR}.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 그래프 2 저장 완료')

In [ ]:
# ── 그래프 3: 연도별 1화기 예보 발생일 비교 ─────────────────
# 관측소별·연도별 1화기 주의 발생 Julian 추출
summary = []
for (stn, year), grp in df.groupby(['stn_nm','year']):
    hit = grp[grp['cumdd'] >= 188.7]
    if len(hit) > 0:
        summary.append({'stn_nm': stn, 'year': year, 'jld_1화기주의': int(hit.iloc[0]['jld'])})

smdf = pd.DataFrame(summary)

fig, ax = plt.subplots(figsize=(13, 6))

colors = {'제주':'#1565C0','서귀포':'#C62828','성산':'#2E7D32','고산':'#F57F17'}
years = sorted(smdf['year'].unique())
x = range(len(years))
width = 0.2

for idx, stn in enumerate(sorted(smdf['stn_nm'].unique())):
    vals = [smdf[(smdf['stn_nm']==stn)&(smdf['year']==y)]['jld_1화기주의'].values[0]
            if len(smdf[(smdf['stn_nm']==stn)&(smdf['year']==y)]) > 0 else None
            for y in years]
    ax.bar([xi + idx*width for xi in x], vals, width=width,
           label=stn, color=colors.get(stn,'gray'), alpha=0.85)

# 권장 범위 표시
ax.axhspan(120, 135, alpha=0.15, color='green', label='논문 권장 범위 (J120~135)')
ax.axhline(y=120, color='green', linestyle='--', linewidth=1.5, alpha=0.7)
ax.axhline(y=135, color='green', linestyle='--', linewidth=1.5, alpha=0.7)

ax.set_xlabel('연도')
ax.set_ylabel('Julian Date (1화기 주의 발생일)')
ax.set_title('네눈쑥가지나방 1화기 주의 발생일 연도별 비교 (관측소별)', fontsize=13, fontweight='bold')
ax.set_xticks([xi + width*1.5 for xi in x])
ax.set_xticklabels(years)
ax.set_ylim(70, 150)
ax.set_yticks([80,90,100,110,120,130,140,150])
ax.set_yticklabels(['3/21','4/1','4/10','4/20','5/1','5/11','5/20','5/31'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/graph3_yearly_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 그래프 3 저장 완료')

In [ ]:
print('='*60)
print('  ✅ STEP 5 완료')
print('='*60)
print(f'  저장 위치: {SAVE_DIR}')
print('  graph1_cumdd_2024.png       — 누적 DD + 위험도 구간')
print('  graph2_sigmoid_제주_2024.png — Sigmoid 개체군 곡선')
print('  graph3_yearly_comparison.png — 연도별 발생일 비교')
print()
print('  → 교수 미팅 자료로 활용 가능')
print('  → 임계값 보정 후 재실행 가능 (CELL 2 파라미터만 수정)')